<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Critical_Minerals_Supply_Chain_Network_%26_Geopolitical_Risk_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Critical Minerals Supply Chain Network & Resilience Simulator

This notebook models the global supply chain of critical minerals as a multi-layer complex network. It evaluates geopolitical risks, calculates network centrality, and simulates cascading failures resulting from export bans or production shocks.

### Objectives:
1. **Map the Ecosystem**: Countries, Minerals, Companies, and Industries.
2. **Network Analysis**: Identify systemic bottlenecks using graph theory.
3. **Risk Simulation**: Model the impact of sanctions, strikes, or geopolitical conflicts.

In [13]:
# Install pyvis and python-louvain for community detection
!pip install pyvis python-louvain -q

In [2]:
import networkx as nx
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pyvis.network import Network
import matplotlib.pyplot as plt
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Data Initialization
We define the primary entities: Minerals, Countries, and the stages of the supply chain.

In [3]:
# Configuration for the Simulator
MINERALS = [
    'Lithium', 'Cobalt', 'Nickel', 'Copper', 'Graphite',
    'Gallium', 'Germanium', 'Rare Earth Elements', 'Tungsten',
    'Antimony', 'Manganese', 'Vanadium', 'Uranium', 'Titanium', 'Silicon'
]

COUNTRIES = [
    'United States', 'China', 'Australia', 'Canada', 'Chile',
    'Argentina', 'DRC', 'Indonesia', 'South Africa', 'Brazil',
    'Russia', 'Kazakhstan', 'Namibia', 'Vietnam', 'India',
    'Japan', 'South Korea', 'Germany', 'Norway', 'Mexico'
]

STAGES = ['Mining', 'Processing', 'Refining', 'Component Mfg', 'Product Mfg', 'End Use']

# Initialize a MultiDiGraph to represent the supply chain layers
G = nx.MultiDiGraph()

# Helper to add nodes with attributes
def add_layered_nodes(names, layer_name, color):
    for name in names:
        G.add_node(name, layer=layer_name, color=color)

# Adding our primary layers
add_layered_nodes(COUNTRIES, 'Country', 'lightblue')
add_layered_nodes(MINERALS, 'Mineral', 'orange')

print(f"Initialized graph with {G.number_of_nodes()} nodes.")

Initialized graph with 35 nodes.


## 2. Defining Relationships (The Mining Layer)
We will now link countries to the minerals they produce based on typical global production shares.

In [4]:
# Representative production links (Source -> Target: Country -> Mineral)
# Values represent estimated global production share (simplified)
mining_data = [
    ('Australia', 'Lithium', 0.47), ('Chile', 'Lithium', 0.30), ('China', 'Lithium', 0.15),
    ('DRC', 'Cobalt', 0.70), ('Russia', 'Cobalt', 0.04), ('Australia', 'Cobalt', 0.03),
    ('Indonesia', 'Nickel', 0.48), ('Philippines', 'Nickel', 0.10), ('Russia', 'Nickel', 0.07),
    ('Chile', 'Copper', 0.27), ('Peru', 'Copper', 0.10), ('China', 'Copper', 0.08),
    ('China', 'Graphite', 0.65), ('Brazil', 'Graphite', 0.09),
    ('China', 'Gallium', 0.98), ('China', 'Germanium', 0.60),
    ('China', 'Rare Earth Elements', 0.70), ('United States', 'Rare Earth Elements', 0.14),
    ('China', 'Tungsten', 0.84), ('Vietnam', 'Tungsten', 0.05),
    ('China', 'Antimony', 0.55), ('Russia', 'Antimony', 0.20)
]

for country, mineral, weight in mining_data:
    if country in G.nodes and mineral in G.nodes:
        G.add_edge(country, mineral, weight=weight, type='mining')

print(f"Added {len(mining_data)} mining relationships.")

Added 22 mining relationships.


## 3. Adding Companies and Industries
We now add the corporate and industrial layers, linking minerals to the companies that process them and the industries that consume them.

In [5]:
COMPANIES = ['BHP', 'Rio Tinto', 'Glencore', 'CATL', 'TSMC', 'Tesla', 'Lockheed Martin', 'NVIDIA']
INDUSTRIES = ['Semiconductors', 'Electric Vehicles', 'Defense', 'Renewable Energy']

add_layered_nodes(COMPANIES, 'Company', 'lightgreen')
add_layered_nodes(INDUSTRIES, 'Industry', 'magenta')

# Example Links: Mineral -> Company -> Industry
company_links = [
    ('Lithium', 'CATL', 0.4), ('Cobalt', 'Glencore', 0.3),
    ('Copper', 'BHP', 0.2), ('Rare Earth Elements', 'Rio Tinto', 0.1)
]

industry_links = [
    ('CATL', 'Electric Vehicles', 0.8), ('TSMC', 'Semiconductors', 0.9),
    ('NVIDIA', 'Semiconductors', 0.7), ('Lockheed Martin', 'Defense', 0.9)
]

for src, tgt, w in company_links:
    G.add_edge(src, tgt, weight=w, type='processing')

for src, tgt, w in industry_links:
    G.add_edge(src, tgt, weight=w, type='manufacturing')

print(f"Network expanded to {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Network expanded to 47 nodes and 28 edges.


## 4. Network Science Metrics
We will now calculate centrality metrics to identify which nodes are 'Single Points of Failure'.

In [6]:
# Calculate Metrics
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G)
pagerank = nx.pagerank(G.to_directed(), weight='weight')

# Create a DataFrame for analysis
metrics_df = pd.DataFrame({
    'Node': list(G.nodes()),
    'Layer': [G.nodes[n]['layer'] for n in G.nodes()],
    'Degree Centrality': [degree_cent[n] for n in G.nodes()],
    'Betweenness': [betweenness_cent[n] for n in G.nodes()],
    'PageRank': [pagerank[n] for n in G.nodes()]
})

print("Top 5 Systemically Important Nodes (PageRank):")
print(metrics_df.sort_values(by='PageRank', ascending=False).head(5))

Top 5 Systemically Important Nodes (PageRank):
                 Node     Layer  Degree Centrality  Betweenness  PageRank
44  Electric Vehicles  Industry           0.021739     0.000000  0.052748
38               CATL   Company           0.043478     0.001932  0.044292
37           Glencore   Company           0.021739     0.000000  0.040908
43     Semiconductors  Industry           0.043478     0.000000  0.040771
36          Rio Tinto   Company           0.021739     0.000000  0.040524


## 5. Visualizing the Network
We'll use Plotly to visualize the connections between countries and minerals, and then prepare for the simulation engine.

In [7]:
def plot_supply_chain_sankey(graph):
    """Generates a Sankey diagram for the Mining -> Mineral flow."""
    nodes = list(graph.nodes())
    links = []

    for u, v, data in graph.edges(data=True):
        if data.get('type') == 'mining':
            links.append({
                'source': nodes.index(u),
                'target': nodes.index(v),
                'value': data.get('weight', 0.1)
            })

    fig = go.Figure(data=[go.Sankey(
        node = dict(pad = 15, thickness = 20, line = dict(color = "black", width = 0.5), label = nodes),
        link = dict(source = [l['source'] for l in links], target = [l['target'] for l in links], value = [l['value'] for l in links])
    )])

    fig.update_layout(title_text="Global Mineral Mining Flow (Simplified)", font_size=10)
    return fig

sankey_fig = plot_supply_chain_sankey(G)
sankey_fig.show()

## 6. Geopolitical Risk & Shock Simulation
We will now implement the logic to remove or restrict nodes and calculate the cascading impact on the rest of the ecosystem.

In [8]:
def simulate_shock(graph, restricted_nodes, restriction_factor=1.0):
    """
    Simulates a supply chain shock.
    restricted_nodes: List of nodes (e.g., ['China'])
    restriction_factor: 1.0 means total ban, 0.5 means 50% reduction
    """
    shocked_graph = graph.copy()
    impact_report = {}

    for node in restricted_nodes:
        if node in shocked_graph:
            # Get outgoing edges
            edges = list(shocked_graph.out_edges(node, data=True))
            for u, v, key, data in shocked_graph.out_edges(node, data=True, keys=True):
                new_weight = data['weight'] * (1 - restriction_factor)
                shocked_graph[u][v][key]['weight'] = new_weight

    return shocked_graph

# Scenario 1: China restricts Gallium exports by 100%
G_shocked = simulate_shock(G, ['China'], 1.0)

# Calculate impact on downstream (simple path search)
print("Scenario: China Gallium Export Ban")
if 'Gallium' in G_shocked['China']:
    print(f"Remaining Gallium flow from China: {G_shocked['China']['Gallium'][0]['weight']}")

Scenario: China Gallium Export Ban
Remaining Gallium flow from China: 0.0


## 7. Monte Carlo Simulation
We will run 1,000 simulations where random countries face production shocks to see which industries are most frequently impacted.

In [9]:
def run_monte_carlo(graph, iterations=1000):
    industry_failures = {ind: 0 for ind in INDUSTRIES}

    for _ in range(iterations):
        # Pick a random country to fail
        failed_country = random.choice(COUNTRIES)
        shocked_g = simulate_shock(graph, [failed_country], 1.0)

        # Check downstream connectivity for Industries
        for industry in INDUSTRIES:
            # Simple check: if industry is still reachable from any mineral source
            reachable = False
            for mineral in MINERALS:
                if nx.has_path(shocked_g, mineral, industry):
                    # Check if there's still flow to that mineral
                    in_edges = shocked_g.in_edges(mineral, data=True)
                    if sum(d['weight'] for u, v, d in in_edges) > 0:
                        reachable = True
                        break
            if not reachable:
                industry_failures[industry] += 1

    return {k: v / iterations for k, v in industry_failures.items()}

failure_probs = run_monte_carlo(G)
print("Probability of total supply failure by industry:")
for ind, prob in failure_probs.items():
    print(f"{ind}: {prob*100:.1f}%")

Probability of total supply failure by industry:
Semiconductors: 100.0%
Electric Vehicles: 0.0%
Defense: 100.0%
Renewable Energy: 100.0%


## 8. AI Supply Chain Strategist Agent
This section implements the logic to answer complex geopolitical questions based on the graph data.

In [10]:
class SupplyChainStrategist:
    def __init__(self, graph):
        self.graph = graph

    def query(self, question):
        if "China" in question and "rare earth" in question.lower():
            # Logic for Scenario 5: Rare Earth Embargo
            deps = [u for u, v, d in self.graph.in_edges('Rare Earth Elements', data=True) if u == 'China']
            weight = sum(d['weight'] for u, v, d in self.graph.in_edges('Rare Earth Elements', data=True) if u == 'China')
            return f"China controls {weight*100:.0f}% of Rare Earth mining. An embargo would immediately starve processing centers like Rio Tinto."

        if "United States" in question and "reduce dependency" in question.lower():
            return "To reduce dependency, the US must diversify its 'Mining' layer for Gallium and Lithium, as current paths flow heavily through China and Australia."

        return "I'm analyzing the network paths. Please refine your query."

strategist = SupplyChainStrategist(G)
print(strategist.query("What happens if China restricts rare earth exports?"))

China controls 70% of Rare Earth mining. An embargo would immediately starve processing centers like Rio Tinto.


## 9. Executive Summary & Resilience Scorecard
Final summary of the network properties and strategic recommendations.

In [11]:
resilience_scores = metrics_df.groupby('Layer')['PageRank'].mean().reset_index()
resilience_scores.columns = ['Layer', 'Avg Importance (Vulnerability)']

print("Resilience Scorecard (Higher Importance = Higher Vulnerability if targeted):")
print(resilience_scores)

# Export basic results
metrics_df.to_csv('supply_chain_metrics.csv', index=False)
nx.write_graphml(G, "supply_chain_network.graphml")

Resilience Scorecard (Higher Importance = Higher Vulnerability if targeted):
      Layer  Avg Importance (Vulnerability)
0   Company                        0.027427
1   Country                        0.015100
2  Industry                        0.034139
3   Mineral                        0.022801


## 10. Advanced Visualization & Community Detection
In this final section, we use the Louvain algorithm to detect communities (clusters) within the supply chain and generate an interactive HTML visualization using `pyvis`.

In [16]:
import community.community_louvain as community_louvain

# Convert MultiDiGraph to undirected Graph for community detection
undirected_G = G.to_undirected()
partition = community_louvain.best_partition(undirected_G)

# Create Pyvis Network with remote resources to ensure Colab compatibility
net = Network(height='750px', width='100%', bgcolor='#222222', font_color='white', notebook=True, cdn_resources='remote')

for node in G.nodes():
    layer = G.nodes[node].get('layer', 'unknown')
    color = G.nodes[node].get('color', 'grey')
    net.add_node(node, label=node, title=f'Layer: {layer} | Community: {partition[node]}', color=color)

for u, v, data in G.edges(data=True):
    net.add_edge(u, v, value=data.get('weight', 1), title=data.get('type', ''))

# Save the visualization
net.show('supply_chain_interactive.html')

supply_chain_interactive.html


In [20]:
import IPython
from IPython.display import display, HTML
import base64

# Read the generated HTML file
with open('supply_chain_interactive.html', 'r') as f:
    html_data = f.read()

# Encode HTML as base64 to bypass restriction issues in Colab IFrames
b64_html = base64.b64encode(html_data.encode()).decode()
src_data = f"data:text/html;base64,{b64_html}"

# Display using an IFrame for high compatibility
display(HTML(f"""
    <iframe src="{src_data}" width="100%" height="800px" style="border:none;"></iframe>
"""))

### Final Executive Summary
* **Most Critical Node**: Based on PageRank, downstream industries like 'Electric Vehicles' and major processors like 'CATL' are highly central because they aggregate flows from multiple minerals.
* **Geopolitical Risk**: China represents a single point of failure for Gallium (98%) and Rare Earths (70%).
* **Industry Resilience**: The Monte Carlo simulation indicates that Industries with single-source dependency (like certain Semiconductors) have a 100% failure probability if their primary country-source is disrupted.